In [1]:
# Importing Libraries

import pandas as pd
import numpy as np
import sklearn
from sklearn import preprocessing
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import statsmodels.api as sm
from statsmodels.formula.api import logit
from statsmodels.stats.outliers_influence import variance_inflation_factor
import seaborn as sns 
import matplotlib.pyplot as plt 
%matplotlib inline
from scipy import stats
import statistics
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import the online purchasing csv file to be used. 
# View dataset to ensure proper loading

df = pd.read_csv('onlinepurchasing.csv')
pd.set_option('display.max_columns', None)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'onlinepurchasing.csv'

In [ ]:
df.info()

In [ ]:
# Find duplicates
df.duplicated()

In [ ]:
df.drop_duplicates(keep='last', inplace=True)

In [ ]:
df.shape

In [ ]:
# Locate missing values in the dataset
df.isnull().sum()

In [ ]:
# Find outliers in the dataset using the IQR method
def find_outliers(df, var):
    q1 = df[var].quantile(0.25)
    q3 = df[var].quantile(0.75)
    IQR = q3 - q1
    lowerbound = q1-(1.5*IQR)
    upperbound = q3+(1.5*IQR)
    outliers = df[var][((df[var] < (lowerbound)) | (df[var] > (upperbound)))]
    return outliers

# Run the function for each variable in the dataset to find outliers and print the number of outliers, max outlier value, and min outlier value.

outliers = find_outliers(df, 'Administrative')
print("number of outliers in Administrative: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'Administrative_Duration')
print("number of outliers in Administrative_Duration: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'Informational')
print("number of outliers in Informational: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'Informational_Duration')
print("number of outliers in Informational_Duration: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'ProductRelated')
print("number of outliers in ProductRelated: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'ProductRelated_Duration')
print("number of outliers in ProductRelated_Duration: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'BounceRates')
print("number of outliers in BounceRates: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'ExitRates')
print("number of outliers in ExitRates: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'PageValues')
print("number of outliers in PageValues: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

outliers = find_outliers(df, 'SpecialDay')
print("number of outliers in SpecialDay: "+ str(len(outliers)))
print("max outlier value: "+ str(outliers.max()))
print("min outlier value: "+ str(outliers.min()))

In [ ]:
# Visualize all variables showing outliers
boxplot=sns.boxplot(x='Administrative',data=df)
plt.show()
boxplot=sns.boxplot(x='Administrative_Duration',data=df)
plt.show()
boxplot=sns.boxplot(x='Informational',data=df)
plt.show()
boxplot=sns.boxplot(x='Informational_Duration',data=df)
plt.show()
boxplot=sns.boxplot(x='ProductRelated',data=df)
plt.show()
boxplot=sns.boxplot(x='ProductRelated_Duration',data=df)
plt.show()
boxplot=sns.boxplot(x='BounceRates',data=df)
plt.show()
boxplot=sns.boxplot(x='ExitRates',data=df)
plt.show()
boxplot=sns.boxplot(x='PageValues',data=df)
plt.show()
boxplot=sns.boxplot(x='SpecialDay',data=df)
plt.show()

In [ ]:
# Treating outliers found.

def find_boundary(df, var):
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    IQR = Q3-Q1
    lower = Q1-(1.5*IQR)
    upper = Q3+(1.5*IQR)
    return lower , upper

lower_adm, upper_adm = find_boundary(df, 'Administrative' )
df.Administrative = np.where(df.Administrative > upper_adm, upper_adm,
                               np.where(df.Administrative < lower_adm, lower_adm, df.Administrative))

lower_adur, upper_adur = find_boundary(df, 'Administrative_Duration' )
df.Administrative_Duration = np.where(df.Administrative_Duration > upper_adur, upper_adur,
                               np.where(df.Administrative_Duration < lower_adur, lower_adur, df.Administrative_Duration))

lower_pr, upper_pr = find_boundary(df, 'ProductRelated' )
df.ProductRelated = np.where(df.ProductRelated > upper_pr, upper_pr,
                               np.where(df.ProductRelated < lower_pr, lower_pr, df.ProductRelated))

lower_prd, upper_prd = find_boundary(df, 'ProductRelated_Duration' )
df.ProductRelated_Duration = np.where(df.ProductRelated_Duration > upper_prd, upper_prd,
                               np.where(df.ProductRelated_Duration < lower_prd, lower_prd, df.ProductRelated_Duration))

lower_bou, upper_bou = find_boundary(df, 'BounceRates' )
df.BounceRates = np.where(df.BounceRates > upper_bou, upper_bou,
                               np.where(df.BounceRates < lower_bou, lower_bou, df.BounceRates))

lower_ex, upper_ex = find_boundary(df, 'ExitRates' )
df.ExitRates = np.where(df.ExitRates > upper_ex, upper_ex,
                               np.where(df.ExitRates < lower_ex, lower_ex, df.ExitRates))

In [ ]:
# Show updated boxplots after treating outliers
boxplot=sns.boxplot(x='Administrative',data=df)
plt.show()
boxplot=sns.boxplot(x='Administrative_Duration',data=df)
plt.show()
boxplot=sns.boxplot(x='Informational',data=df)
plt.show()
boxplot=sns.boxplot(x='Informational_Duration',data=df)
plt.show()
boxplot=sns.boxplot(x='ProductRelated',data=df)
plt.show()
boxplot=sns.boxplot(x='ProductRelated_Duration',data=df)
plt.show()
boxplot=sns.boxplot(x='BounceRates',data=df)
plt.show()
boxplot=sns.boxplot(x='ExitRates',data=df)
plt.show()
boxplot=sns.boxplot(x='PageValues',data=df)
plt.show()
boxplot=sns.boxplot(x='SpecialDay',data=df)
plt.show()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
df.columns

In [ ]:
# Locating cardinality of variables
print(f'Administrative: {df.Administrative.nunique()}')
print(f'Administrative_Duration: {df.Administrative_Duration.nunique()}')
print(f'Informational: {df.Informational.nunique()}')
print(f'Informational_Duration: {df.Informational_Duration.nunique()}')
print(f'ProductRelated: {df.ProductRelated.nunique()}')
print(f'ProductRelated_Duration: {df.ProductRelated_Duration.nunique()}')
print(f'BounceRates: {df.BounceRates.nunique()}')
print(f'ExitRates: {df.ExitRates.nunique()}')
print(f'PageValues: {df.PageValues.nunique()}')
print(f'SpecialDay: {df.SpecialDay.nunique()}')
print(f'Month: {df.Month.nunique()}')
print(f'OperatingSystems: {df.OperatingSystems.nunique()}')
print(f'Browser: {df.Browser.nunique()}')
print(f'Region: {df.Region.nunique()}')
print(f'TrafficType: {df.TrafficType.nunique()}')
print(f'VisitorType: {df.VisitorType.nunique()}')
print(f'Weekend: {df.Browser.nunique()}')
print(f'Revenue: {df.Region.nunique()}')

In [ ]:
# Drop categorical, unneeded variables
df.drop(['Month','OperatingSystems', 'Browser', 'Region', 'TrafficType'], axis=1, inplace=True)

In [ ]:
# Univariate statistics, via distplots for quantitative explanatory variables

sns.displot(df['Administrative'], kde=False, color='blue', bins=7)
sns.displot(df['Administrative_Duration'], kde=False, color='red', bins=7)
sns.displot(df['Informational'], kde=False, color='green', bins=7)
sns.displot(df['Informational_Duration'], kde=False, color='gray', bins=7)
sns.displot(df['ProductRelated'], kde=False, color='orange', bins=7)
sns.displot(df['ProductRelated_Duration'], kde=False, color='black', bins=5)
sns.displot(df['BounceRates'], kde=False, color='purple', bins=5)
sns.displot(df['ExitRates'], kde=False, color='pink', bins=7)
sns.displot(df['PageValues'], kde=False, color='brown', bins=7)
sns.displot(df['SpecialDay'], kde=False, color='white', bins=7)

In [ ]:
# Univariate statistics, via barplots for categorical explanatory variables

groupedWeekEnd = df.groupby(by='Weekend').size()
groupedWeekEnd
%matplotlib inline
groupedWeekEnd.plot.bar()

In [ ]:
groupedVisitor = df.groupby(by='VisitorType').size()
groupedVisitor
%matplotlib inline
groupedVisitor.plot.bar()

In [ ]:
groupedRevenue = df.groupby(by='Revenue').size()
groupedRevenue
%matplotlib inline
groupedRevenue.plot.bar()

In [ ]:
# Bivariate statistics visualizations 
# 2 categorical variables using crosstab table and barplot
 
rv_crosstab=pd.crosstab(index=df['Revenue'], columns=df['VisitorType'])
print(rv_crosstab)

In [ ]:
rv_crosstab.plot.bar(figsize=(7,4), rot=0)

In [ ]:
rw_crosstab=pd.crosstab(index=df['Revenue'], columns=df['Weekend'])
print(rw_crosstab)

In [ ]:
rw_crosstab.plot.bar(figsize=(7,4), rot=0)

In [ ]:
# Bivariate statistics of categorical/continuous variable via boxplots

sns.boxplot(data=df, x="Revenue", y="Administrative")


In [ ]:
sns.boxplot(data=df, x="Revenue", y="Administrative_Duration")

In [ ]:
sns.boxplot(data=df, x="Revenue", y="Informational")

In [ ]:
sns.boxplot(data=df, x="Revenue", y="Informational_Duration")

In [ ]:
sns.boxplot(data=df, x="Revenue", y="ProductRelated")

In [ ]:
sns.boxplot(data=df, x="Revenue", y="ProductRelated_Duration")

In [ ]:
sns.boxplot(data=df, x="Revenue", y="BounceRates")

In [ ]:
sns.boxplot(data=df, x="Revenue", y="ExitRates")

In [ ]:
sns.boxplot(data=df, x="Revenue", y="PageValues")

In [ ]:
sns.boxplot(data=df, x="Revenue", y="SpecialDay")

In [ ]:
# Re-expression of categorical variables
# Convert ordinal categorical to numerical

df['Weekend']=df['Weekend'].astype('category')
df['Weekend']=df['Weekend'].cat.codes
df['Revenue']=df['Revenue'].astype('category')
df['Revenue']=df['Revenue'].cat.codes

In [ ]:
# Utilizing get_dummies to convert nominal categorical to numerical
df= pd.get_dummies(df, columns=['VisitorType'], prefix_sep=" " , drop_first=True)

In [ ]:
df.info()

In [ ]:
X=df[['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Weekend',
       'VisitorType Other', 'VisitorType Returning_Visitor']]

vif_data = pd.DataFrame()
vif_data["Explanatory Variables"] = X.columns

vif_data["VIF"] = [variance_inflation_factor(X.astype(float).values, i)
                               for i in range(len(X.columns))]

vif_data["VIF"]=round(vif_data["VIF"],2)
vif_data=vif_data.sort_values(by="VIF", ascending=False)

print(vif_data)

In [ ]:
df.describe()

In [ ]:
# Logistic regression model
df['const']=1
y= df['Revenue']
X= df[['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Weekend',
       'VisitorType Other', 'VisitorType Returning_Visitor','const']]
log_model = sm.Logit(y.astype(float), X.astype(float))
results = log_model.fit()
print(results.summary())

In [ ]:
y= df['Revenue'] 
X= df[['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Weekend',
       'VisitorType Other', 'VisitorType Returning_Visitor','const']]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

In [ ]:
scaler = RobustScaler()

In [ ]:
X_train = scaler.fit_transform(X_train)

In [ ]:
X_test= scaler.transform(X_test)

In [ ]:
log_r= LogisticRegression(random_state=0).fit(X_train, y_train)

In [ ]:
y_pred= log_r.predict(X_test)

In [ ]:
log_r.score(X_train, y_train)

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
matrix_i = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(matrix_i)
ax.grid(False)
ax.xaxis.set(ticks=(0, 1), ticklabels=('Predicted 0s', 'Predicted 1s'))
ax.yaxis.set(ticks=(0, 1), ticklabels=('Actual 0s', 'Actual 1s'))
ax.set_ylim(1.5, -0.5)
for i in range(2):
    for j in range(2):
        ax.text(j, i, matrix_i[i, j], ha='center', va='center', color='orange')
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
# Reduced model via backward stepwise elmination with p-values less than 0.05

df['const']=1
y= df['Revenue']
X=df[['ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Weekend',
       'VisitorType Returning_Visitor','const']]
log_model_red = sm.Logit(y.astype(float), X.astype(float))
redu_results = log_model_red.fit()
print(redu_results.summary())

In [ ]:
y= df['Revenue']
X=df[['ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Weekend',
       'VisitorType Returning_Visitor','const']]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

In [ ]:
scaler = RobustScaler()

In [ ]:
X_train = scaler.fit_transform(X_train)

In [ ]:
X_test= scaler.transform(X_test)

In [ ]:
logr= LogisticRegression(random_state=0).fit(X_train, y_train)

In [ ]:
logr.score(X_test, y_test)

In [ ]:
y_predr= logr.predict(X_test)

In [ ]:
confusion_matrix(y_test, y_predr)

In [ ]:
matrix = confusion_matrix(y_test, y_predr)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(matrix)
ax.grid(False)
ax.xaxis.set(ticks=(0, 1), ticklabels=('Predicted 0s', 'Predicted 1s'))
ax.yaxis.set(ticks=(0, 1), ticklabels=('Actual 0s', 'Actual 1s'))
ax.set_ylim(1.5, -0.5)
for i in range(2):
    for j in range(2):
        ax.text(j, i, matrix[i, j], ha='center', va='center', color='red')
plt.show()

In [ ]:
print(classification_report(y_test, y_predr))

In [ ]:
df.to_csv(r'onlinepurchasing_clean.csv')